## **Task: Evading AI-Generated Text Detection**

On the image below you can see how distributions of generations of gemma-2 human texts and gpt look like. Your goal is to shift gemma distribution to the right, so that it becomes similar to the human's, but not to gpt (so you need to shift it to the right, but not too much).

![](https://i.postimg.cc/Kzjk7Kc4/IMG-7240.jpg)

## Baseline

First, we login to HuggingFace and import required modules

You need to accept the agreement of [gemma2-2b](https://huggingface.co/google/gemma-2-2b).

Keep in mind that this model is not able to infer on kaggle GPU due to not enough memory in P100 GPU.

In [ ]:
from huggingface_hub import login

login("hf_XXXXXXXXXXX")

In [ ]:
import sys
# sys.path.append("C:\\Users\\raian\\.cache\\kagglehub\\datasets\\egorgij21\\neoai-2025-dftd-baseline-code\\versions\\1")

import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import matplotlib.pyplot as plt
from dataclasses import dataclass

import torch
from torch.utils.data import DataLoader

from dataset import FakeTextDataset
from detector import FakeTextDetector

As an example of pre-trained model that you can use for understanding how to change weights/representation of gemma, we can use SAE trained for this model. Let's install the package for it:

In [ ]:
from sae_lens import SAE, HookedSAETransformer

Define config for gemma-2-2b. You can change any parameter here except `model_name`

In [ ]:
@dataclass
class Config:
    # data params
    data_path: str = "/kaggle/input/neoai-2025-deception-fake-text-detector"
    test_dataset_name: str = "test.csv"
    num_workers: int = 1
    batch_size: int = 20
    output_submission_path: str = "submission.csv"

    # gemma params
    device_llm: str = "cuda:0"
    model_name = "google/gemma-2-2b"

    # sae params
    release: str = "gemma-scope-2b-pt-res-canonical"
    device_sae: str = "cuda:0"
    layer: int = 20
    num_latents_k: int = 16

    # detector params
    device_detector: str = "cuda:0"

Now we initialize the models:

In [ ]:
# Initialize Gemma model
model = HookedSAETransformer.from_pretrained(Config.model_name, local_files_only=False, device=Config.device_llm)
model.eval()

# Initialize SAE model
sae, _, _ = SAE.from_pretrained(
    release=Config.release,
    sae_id=f"layer_{Config.layer}/width_{Config.num_latents_k}k/canonical"
)
sae = sae.to(Config.device_sae)

# Initialize detector model
detector = FakeTextDetector(device=Config.device_detector)

## Inference

In the cell below there is infer function that takes model and some arguments and generates submission. The code of this function cannot be changed (however arguments can be changed).

In [ ]:
def infer(
        config: Config,
        model: torch.nn.Module,
        max_new_tokens: int = 128,
        stop_at_eos: bool = True,
        prepend_bos: bool = True,
        verbose: bool = False,
        skip_special_tokens: bool = True
    ) -> None:
    dataset = FakeTextDataset(os.path.join(config.data_path, config.test_dataset_name), mode="test")
    dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=False)

    submission = {"prompt": [], "generation": []}
    for batch in tqdm(dataloader):
        prompts = batch["prompt"]

        submission["prompt"].extend(prompts)

        with torch.no_grad():
            input_ids = model.to_tokens(prompts, prepend_bos=True)
            output = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                stop_at_eos=stop_at_eos,
                prepend_bos=prepend_bos,
                verbose=verbose
            ).cpu().numpy()
        generated_texts = model.tokenizer.batch_decode(output, skip_special_tokens=skip_special_tokens)
        submission["generation"].extend(generated_texts)

    submission = pd.DataFrame(submission)

    submission.prompt = submission.prompt.apply(lambda x: x.replace('"', "'"))
    submission.generation = submission.generation.apply(lambda x: x.replace('"', '"'))

    submission = submission.astype(pd.StringDtype())

    submission.to_csv(config.output_submission_path, index=False)


Note: we infer metric on cpu and have 30 minutes for submission. So don't choose large `max_new_tokens` parameter (512+)

In [ ]:
infer(config=Config, model=model, max_new_tokens=128)

## Metric

In the cell below there is metric code which will be infered on kaggle.

Metric has two components:
- binary classifier $BC$ with thr=0.65 which determines how much the output of modified gemma is changed compared to the output of unchanged base model. If similarity < 0.65, the text is considered to be changed too much and the overall metric for this text will be 0. So you need to make sure that the outputs of your changed model are similar enough to that of the unchanged model in terms of this classifier;
- score of the fake text detector model $FTDM$. This model taked gemma generation as input and outputs a reward score $R$. If $R$ is between score_threshold_low and score_threshold_high (and the first part of the metric returns 1 for this text), the score of the text is 1, otherwise it is 0. Thus the overall metric for the collection of generated text is an accuracy computed using fake text detector model $FTDM$ multiplied by a mask provided by binary classifier $BC$.

In [ ]:
from typing import List, Union

from sentence_transformers import SentenceTransformer, util

from transformers import AutoModelForSequenceClassification, AutoTokenizer


class TextComparator:
    def __init__(
        self,
        device: str = "cuda:0",
        model_dir: str = "/kaggle/input/text-comparator"
    ) -> None:
        self.model = SentenceTransformer(model_dir, device=device)
        self.embedding_cache = {}

    def get_embeddings(self, texts: Union[str, List[str]]) -> np.ndarray:
        single_text = isinstance(texts, str)
        if single_text:
            texts = [texts]

        uncached_texts = [t for t in texts if t not in self.embedding_cache]
        if uncached_texts:
            new_embeddings = self.model.encode(uncached_texts, convert_to_numpy=True)
            for text, embedding in zip(uncached_texts, new_embeddings):
                self.embedding_cache[text] = embedding

        embeddings = np.array([self.embedding_cache[t] for t in texts])

        return embeddings[0] if single_text else embeddings

    def similarity(
        self,
        text1: Union[str, List[str]],
        text2: Union[str, List[str]]
    ) -> Union[float, np.ndarray]:
        emb1 = self.get_embeddings(text1)
        emb2 = self.get_embeddings(text2)

        return np.diag(util.cos_sim(emb1, emb2).numpy())


class FakeTextDetector:
    def __init__(
        self,
        device: str = "cuda:0",
        model_dir: str = "/kaggle/input/fake-text-detector/"
    ):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        self.model.eval()
        self.model.to(device)

    def get_score(self, text):
        inputs = self.tokenizer(text, return_tensors='pt', truncation=True, max_length=512, padding=True).to(self.device)
        with torch.no_grad():
            score = self.model(**inputs).logits.cpu().detach().numpy()
        return score


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    similarity_threshold: float = 0.65,
    score_threshold_low: float = -2.0,
    score_threshold_high: float = 2.0
) -> float:
    assert solution.shape == submission.shape
    assert (solution[row_id_column_name] == submission[row_id_column_name]).all()

    comparator = TextComparator()
    detector = FakeTextDetector()

    similarities = comparator.similarity(
        solution["generation"].tolist(),
        submission["generation"].tolist()
    )

    steered_texts = submission["generation"].tolist()

    scores = []
    for steered_text, similarity_score in tqdm(zip(steered_texts, similarities)):
        if similarity_score < similarity_threshold:
            scores.append(0)
            continue

        scores_i = detector.get_score(steered_text).item()
        scores.append(scores_i)

    similarity_mask = similarities > similarity_threshold
    deceptions = (np.array(scores) > score_threshold_low) & (np.array(scores) < score_threshold_high) & similarity_mask
    return deceptions.mean()


Let's get the score for gemma-2-2b outputs without any changes, so it'll be our baseline:

In [ ]:
submission = pd.read_csv(Config.output_submission_path)

metric = score(solution=submission, submission=submission, row_id_column_name="prompt")
print(f"LB metric in best case: {metric.item()}")

Good luck! =)